# 03 - Pose Training (Lightning Pose)
Train a pose estimation model on labeled frames using Lightning Pose with GPU acceleration.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Drive paths
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"
DRIVE_LABELED = f"{DRIVE_ROOT}/labeled_frames"
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"

# Training config
BATCH_SIZE = 16
MAX_EPOCHS = 200
LEARNING_RATE = 1e-3
BACKBONE = "resnet50"  # Options: "resnet50", "resnet101", "resnext50"
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

In [ ]:
# Install Lightning Pose (Colab-compatible, GPU version)
# Uses PyTorch with CUDA
!pip install --quiet "lightning-pose[all]" imageio[ffmpeg]

# Verify install
import lightning_pose
print(f"Lightning Pose version: {lightning_pose.__version__}")

In [ ]:
import os
from pathlib import Path

labeled_path = Path(DRIVE_LABELED)
print(f"Labeled frames directory: {labeled_path}")
if labeled_path.exists():
    files = list(labeled_path.rglob("*"))
    images = [f for f in files if f.suffix.lower() in (".jpg", ".png")]
    print(f"Found {len(images)} images")
    # Look for label files (JSON from LabelMe or CSV from LP)
    labels = [f for f in files if f.suffix.lower() in (".csv", ".json", ".h5")]
    print(f"Found {len(labels)} label files")
    for l in labels:
        print(f"  - {l.name}")
else:
    print("Labeled frames directory not found. Complete notebook 02 first.")

In [ ]:
# Convert LabelMe JSON labels to Lightning Pose CSV format (if using LabelMe)
from src.pose.label_converter import convert_labelme_to_lp_csv, validate_labels, KEYPOINT_NAMES

LABEL_CSV = str(labeled_path / "CollectedData_LP.csv")

if labeled_path.exists():
    json_files = list(labeled_path.rglob("*.json"))
    if json_files:
        print(f"Found {len(json_files)} LabelMe JSON files. Converting...")
        csv_path = convert_labelme_to_lp_csv(labeled_path, LABEL_CSV)
        print(f"Labels converted to: {csv_path}")
        validation = validate_labels(csv_path)
        if validation["ready"]:
            print(f"All {len(KEYPOINT_NAMES)} keypoints labeled across {validation['total_images']} images")
        else:
            print(f"Missing keypoints: {validation['keypoints_missing']}")
    else:
        existing_csv = list(labeled_path.rglob("*.csv"))
        if existing_csv:
            LABEL_CSV = str(existing_csv[0])
            print(f"Using existing CSV: {LABEL_CSV}")
        else:
            print("No label files found. Label your frames offline first (see instructions).")

In [ ]:
import yaml

# Create Lightning Pose config
config = {
    "data": {
        "csv_file": LABEL_CSV,
        "image_dir": str(labeled_path),
        "train_fraction": 0.8,
        "test_fraction": 0.1,
        "num_keypoints": 8,
        "keypoint_names": ["snout", "left_ear", "right_ear", "neck",
                           "shoulders", "mid_back", "hip", "tail_base"],
    },
    "training": {
        "backbone": BACKBONE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "gpu_id": 0,
    },
    "inference": {
        "save_heatmaps": False,
    },
}

config_dir = Path("/content/configs")
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "pose_training.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print(f"Config saved to {config_path}")
print("\nIMPORTANT: Edit the 'csv_file' path above to match your labeled data file.")

In [ ]:
from lightning_pose.utils.scripts import train_model

print("Starting training...")
print(f"Backbone: {BACKBONE}, Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}")

# The train_model function expects a config file path
# This will be adapted based on the actual Lightning Pose API
!python -m lightning_pose.train --config {config_path}

print("\nTraining complete!")

In [ ]:
import shutil
from pathlib import Path

models_dir = Path(DRIVE_MODELS)
models_dir.mkdir(parents=True, exist_ok=True)

# Find the latest checkpoint
checkpoint_dir = Path("/content/outputs")
checkpoints = list(checkpoint_dir.rglob("*.ckpt")) + list(checkpoint_dir.rglob("*.pt"))
if checkpoints:
    latest = max(checkpoints, key=os.path.getctime)
    dest = models_dir / "pose_model.ckpt"
    shutil.copy2(str(latest), str(dest))
    print(f"Saved model to {dest}")
else:
    print("No checkpoints found in /content/outputs. Check Lightning Pose output directory.")

In [ ]:
print("=" * 60)
print("Model evaluation metrics will appear here.")
print("Lightning Pose generates evaluation during training.")
print("=" * 60)
print(f"\nModel saved to: {DRIVE_MODELS}/pose_model.ckpt")